# SVM baseline: three training variants

Fits an RBF SVM on the **same partitions the CNN used**, reading
`cnn-latest/results/splits/{variant}-seed{seed}.csv`. `SVC` is deterministic given its
data, so the split is the only source of run-to-run variation and every SVM checkpoint is
*paired* with a CNN checkpoint rather than merely matched in protocol.

| variant | pool | partition |
| --- | --- | --- |
| `orig` | `training.npz` (7,200) | `orig-seed{s}` |
| `clean` | `training-clean.npz` (7,172) | `clean-seed{s}` |
| `clean-fixed` | `training.npz` minus the 28 defective rows | `orig-seed{s}`, rows dropped in place |

`orig` vs `clean` changes the data *and* redraws the split. `clean-fixed` changes only the
data, so it is the arm that isolates the 28 clips as a cause.

`C=1000, gamma=1e-5` are frozen across every fit, inherited from the published tuning, so
the variant comparison is an ablation and not a re-tuning exercise. Fitting uses the
`train` split only, never `train+val`, so the SVM sees exactly the rows the CNN trained
on — which means these runs will *not* reproduce `models/svm-best`, fit on a 90/10 split.

Each fit runs in its own OS process. That is not stylistic: a fitted model holds a 1.35 GB
float64 support-vector array, and process exit is the only thing that reliably returns it.

In [1]:
import subprocess, sys, os, time
from pathlib import Path
import numpy as np, pandas as pd

HERE = Path("/home/seya/code/chord-detection/training/notebooks/svm-latest")
PY = "/home/seya/code/chord-detection/.venv/bin/python"
sys.path.insert(0, str(HERE))
import svm_common as C

C.ensure_dirs()
print("training root:", C.TRAINING_ROOT)
print("variants:", C.VARIANTS)
print("seeds:", C.SEEDS)
print("frozen hyperparameters: C=%g gamma=%g" % (C.DEFAULT_C, C.DEFAULT_GAMMA))

rows = []
for v in C.VARIANTS:
    for tag in ["seed%d" % s for s in C.SEEDS]:
        idx = C.split_indices(C.load_split(v, tag))
        rows.append({"variant": v, "tag": tag,
                     **{k: len(x) for k, x in idx.items()}})
pd.DataFrame(rows).pivot_table(index="variant", columns="tag",
                               values=["train", "val", "test"], aggfunc="first")

training root: /home/seya/code/chord-detection/training
variants: ['orig', 'clean', 'clean-fixed']
seeds: [42, 43, 44, 45, 46]
frozen hyperparameters: C=1000 gamma=1e-05


test                              train                       \
tag         seed42 seed43 seed44 seed45 seed46 seed42 seed43 seed44 seed45   
variant                                                                      
clean          718    718    718    718    718   5736   5736   5736   5736   
clean-fixed    718    717    719    719    716   5736   5736   5736   5741   
orig           720    720    720    720    720   5760   5760   5760   5760   

                      val                              
tag         seed46 seed42 seed43 seed44 seed45 seed46  
variant                                                
clean         5736    718    718    718    718    718  
clean-fixed   5739    718    719    717    712    717  
orig          5760    720    720    720    720    720

## Step zero — prove the fast predict is exact

`SVC.predict` runs single-threaded libsvm, which streams the whole 1.35 GB float64
support-vector array once per clip. At 0.27–0.43 s/clip the planned campaign is ~20 hours
of prediction alone, so `fast_rbf.FastRBF` recomputes the same decision values with one
float32 GEMM per chunk plus 36 small per-class matmuls.

Being *fast* is worthless if it is not *identical*. This cell checks the reconstruction
against the published `models/svm-best` on `vivo` — the hardest available rows, whose
scaled norms are ~1.7× the training set's, so kernel values are smallest and decision
values are closest to a sign flip. It raises on any single disagreement.

In [5]:
r = subprocess.run([PY, str(HERE / "gate_svmbest.py"), "--n", "96"],
                   capture_output=True, text=True, cwd=HERE)
print(r.stdout or r.stderr[-3000:])
assert "GATE PASSED" in r.stdout, "fast predict does not match libsvm - stop here"

svm-best: 36 classes, 4143 SVs, gamma 1e-05, C 1000
libsvm         185 ms/clip
mean max K     0.5084

f32_gemm
  agreement    96/96 exact
  dec error    max 2.364e-07  median 9.546e-09
  min |dec|    8.055e-06  -> margin 34.1x
  speed        1.00 ms/clip  (185x libsvm)

f64_gemm
  agreement    96/96 exact
  dec error    max 1.721e-14  median 3.279e-15
  min |dec|    8.055e-06  -> margin 4.68e+08x
  speed        1.97 ms/clip  (94x libsvm)

GATE PASSED



## First fit — reproduce the known result before trusting anything downstream

`orig` seed 42 should land near the previously recorded `thinkpad-2` accuracy of 0.572
with `C_diminished_4` absorbing ~37% of predictions. It will not match exactly: the
published `svm-best` was fit on a 90/10 split of the whole pool (6,480 rows), while this
fits the CNN's `train` split (5,760 rows). Close is the bar; far means the pipeline
differs from the original run and nothing after this point is trustworthy.

In [6]:
def fit_one(variant, seed, extra=()):
    """One (variant, seed) in its own process; a 1.35 GB SV array is only reclaimed on exit."""
    t = time.time()
    cmd = [PY, str(HERE / "run_one.py"), "--variant", variant, "--seed", str(seed), *extra]
    p = subprocess.Popen(cmd, cwd=HERE, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line.rstrip(), flush=True)
    p.wait()
    print(f"[{variant}/seed{seed} exit {p.returncode} in {time.time() - t:.0f}s]")
    return p.returncode

fit_one("orig", 42)

orig/seed42: train 5760 val 720 test 720


  fit 316.3s  nSV 3893


  gate OK 128 rows, min|dec| 7.569e-06 (30s)


  thinkpad    acc 0.8597  sink C_diminished_4 14.9%


  vivo        acc 0.9750  sink G_minor_4 4.0%


...

  thinkpad-2  acc 0.5569  sink C_diminished_4 38.0%


  flow-2      acc 0.7743  sink C_diminished_4 18.1%


OK orig/seed42  test 1.0000  1.89 ms/clip


[orig/seed42 exit 0 in 423s]


0

## The full sweep

Three variants × five splits. `run_one.py` skips any `(variant, seed)` already in
`progress.csv`, so re-running this cell resumes rather than repeats — `orig` seed 42 above
is already done and will be skipped.

Runs are strictly sequential. Two concurrent fits each hold a multi-GB support-vector
array, and a measured attempt at that degraded libsvm by ~20× through memory-bandwidth
contention alone. Roughly 7 minutes per run.

In [ ]:
t0 = time.time()
for variant in C.VARIANTS:
    for seed in C.SEEDS:
        fit_one(variant, seed)
print(f"\nsweep finished in {(time.time() - t0) / 60:.0f} min")

skip orig/seed42 (already in progress.csv)


[orig/seed42 exit 0 in 1s]


orig/seed43: train 5760 val 720 test 720


  fit 290.9s  nSV 3865


  gate OK 128 rows, min|dec| 2.547e-06 (31s)


...

  thinkpad-2  acc 0.3944  sink C_diminished_4 56.0%


  flow-2      acc 0.6611  sink C_diminished_4 28.1%


OK clean/seed46  test 1.0000  1.61 ms/clip


[clean/seed46 exit 0 in 581s]


clean-fixed/seed42: train 5736 val 718 test 718
